# Multimodal MIMIC-4 Task Tutorial
**Contributors:** William Pang, Joshua Chen, Rian Atri

This notebook demonstrates how to use the `multimodal_mimic4` task.

**Related Resources** (*Note: We likely need to update file paths once merged to Sunlab Pyhealth Main*)
- [Multimodal MIMIC4 Example](https://github.com/Multimodal-PyHealth/PyHealth/blob/main/examples/mortality_prediction/multimodal_mimic4.py)
- [Multimodal MIMIC4 Task](https://github.com/Multimodal-PyHealth/PyHealth/blob/main/pyhealth/tasks/multimodal_mimic4.py)

## 0. Environment Setup and Loading Packages

In [ ]:
from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import shutil 
import random
import numpy as np
import torch
import logging

*Pyhealth Specific Packages*

In [ ]:
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ICDLabsMIMIC4, ClinicalNotesMIMIC4, ClinicalNotesICDLabsMIMIC4, BaseMultimodalMIMIC4Task

*Disable Logging*

In [ ]:
logging.disable(logging.CRITICAL)

*Set Randomness Seed*

In [ ]:
SEED = 25
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

*Mortality Label Filter `(0, 1, or [0, 1])`*

In [ ]:
MORTALITY_LABEL = [1]

## 1. Notebook-Specific Utility Functions

In [ ]:
def filter_mortality_samples(samples, label=None):
    """Return samples filtered by mortality label.

    Args:
        samples: list of sample dicts with a 'mortality' tensor field.
        label: 0, 1, [0, 1], or None to return all samples.
    """
    if label is None or label == [0, 1] or label == [1, 0]:
        return list(samples)
    labels = {label} if isinstance(label, int) else set(label)
    return [s for s in samples if s['mortality'].item() in labels]


def print_patient_admission_info(sample_patient_id):
    patient = dataset.get_patient(sample_patient_id)
    admissions = patient.get_events(
        event_type="admissions",
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\nnum_admissions:  {len(admissions)}")
    for adm in admissions:
        print(f"  hadm_id:       {adm.hadm_id}")
        print(f"  admittime:     {adm.timestamp}")
        print(f"  dischtime:     {adm.dischtime}")


def print_patient_note_info(sample_patient_id, note_type="discharge", char_limit=80):
    patient = dataset.get_patient(sample_patient_id)
    notes = patient.get_events(
        event_type=note_type,
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\n{note_type} notes: {len(notes)}")
    for note in notes:
        print(f"  note_id:   {note.note_id}")
        print(f"  hadm_id:   {note.hadm_id}")
        print(f"  charttime: {note.timestamp}")
        print(f"  storetime: {note.storetime}")
        print(f"  text:      {note.text[:char_limit]}..(Limited to {char_limit} Characters).")


LABCATEGORIES = {
    itemid: name
    for name, itemids in BaseMultimodalMIMIC4Task.LAB_CATEGORIES.items()
    for itemid in itemids
}


def print_patient_lab_info(sample_patient_id):
    patient = dataset.get_patient(sample_patient_id)
    labs = patient.get_events(
        event_type="labevents",
        start=sample['window_start'],
        end=sample['window_end'],
    )
    print(f"\nlabevents: {len(labs)}")
    for lab in labs:
        lab_name = LABCATEGORIES.get(lab['itemid'], "Unknown")
        print(f"  itemid:    {lab['itemid']} ({lab_name})")
        print(f"  charttime: {lab.timestamp}")
        print(f"  storetime: {lab['storetime']}")
        print(f"  valuenum:  {lab['valuenum']}")

def clear_cache_directory(path):
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)
    print(f"Cache directory cleared: {path}")

## 2. Load Demo Dataset

We use the MIMIC-IV demo data in `test-resources/core/mimic4demo`. 

This includes:
- Synthetic Notes (`discharge`, `radiology`)

In [ ]:
PYHEALTH_REPO_ROOT = '/Users/williampang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "test-resources/core/mimic4demo")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "test-resources/core/mimic4demo")
CACHE_DIR = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/wp/demo_pyhealth_cache")

## 3. Multimodal Variants

### 3.1 ICDLabsMIMIC4

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    ehr_tables=["diagnoses_icd", "procedures_icd", "labevents", "prescriptions"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)

task = ICDLabsMIMIC4()
samples = dataset.set_task(task, num_workers=1)

In [ ]:
samples = filter_mortality_samples(samples, label=MORTALITY_LABEL)
sample = samples[0]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")

# Admissions
print_patient_admission_info(sample_patient_id)
# Discharge Notes
print_patient_note_info(sample_patient_id, note_type="discharge")
# Radiology Notes
print_patient_note_info(sample_patient_id, note_type="radiology")
# Labs
print_patient_lab_info(sample_patient_id)

In [ ]:
clear_cache_directory(CACHE_DIR)

### 3.2 ClinicalNotesMIMIC4

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    note_root=NOTE_ROOT,
    note_tables=["discharge", "radiology"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)
task = ClinicalNotesMIMIC4()
samples = dataset.set_task(task, num_workers=1)

In [ ]:
samples = filter_mortality_samples(samples, label=MORTALITY_LABEL)
sample = samples[0]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")

# Admissions
print_patient_admission_info(sample_patient_id)
# Discharge Notes
print_patient_note_info(sample_patient_id, note_type="discharge")
# Radiology Notes
print_patient_note_info(sample_patient_id, note_type="radiology")
# Labs
print_patient_lab_info(sample_patient_id)

In [ ]:
clear_cache_directory(CACHE_DIR)

### 3.3 ClinicalNotesICDLabsMIMIC4

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    note_root=NOTE_ROOT,
    ehr_tables=["diagnoses_icd", "procedures_icd", "labevents", "prescriptions"],
    note_tables=["discharge", "radiology"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)

task = ClinicalNotesICDLabsMIMIC4()
samples = dataset.set_task(task, num_workers=1)

In [ ]:
samples = filter_mortality_samples(samples, label=MORTALITY_LABEL)
sample = samples[0]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")

# Admissions
print_patient_admission_info(sample_patient_id)
# Discharge Notes
print_patient_note_info(sample_patient_id, note_type="discharge")
# Radiology Notes
print_patient_note_info(sample_patient_id, note_type="radiology")
# Labs
print_patient_lab_info(sample_patient_id)

In [ ]:
clear_cache_directory(CACHE_DIR)

## 4. Time Filtering

### 4.1 Full Patient Window
By default the task uses all data between the first admission time and the last discharge time across the patient's processed admissions.

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    note_root=NOTE_ROOT,
    ehr_tables=["diagnoses_icd", "procedures_icd", "labevents"],
    note_tables=["discharge", "radiology"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)

task = ClinicalNotesICDLabsMIMIC4()
samples = dataset.set_task(task, num_workers=1)

In [ ]:
samples = filter_mortality_samples(samples, label=MORTALITY_LABEL)
sample = samples[0]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")

# Admissions
print_patient_admission_info(sample_patient_id)
# Discharge Notes
print_patient_note_info(sample_patient_id, note_type="discharge")
# Radiology Notes
print_patient_note_info(sample_patient_id, note_type="radiology")
# Labs
print_patient_lab_info(sample_patient_id)

In [ ]:
clear_cache_directory(CACHE_DIR)

### 4.2 Using `window_hours`
By using `window_hours`, we only look at data from `first admission + window hours` to predict the mortality task.

In [ ]:
WINDOW_HOURS = 24

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=EHR_ROOT,
    note_root=NOTE_ROOT,
    ehr_tables=["diagnoses_icd", "procedures_icd", "labevents"],
    note_tables=["discharge", "radiology"],
    cache_dir=CACHE_DIR,
    num_workers=1,
)

task = ClinicalNotesICDLabsMIMIC4(window_hours=WINDOW_HOURS)
samples = dataset.set_task(task, num_workers=1)

In [ ]:
samples = filter_mortality_samples(samples, label=MORTALITY_LABEL)
sample = samples[0]
sample_patient_id = sample['patient_id']

print(f"\nPatient ID:      {sample_patient_id}")
print(f"Mortality Flag:  {sample['mortality']}")
print(f"Window start:    {sample['window_start']}")
print(f"Window end:      {sample['window_end']}")

# Admissions
print_patient_admission_info(sample_patient_id)
# Discharge Notes
print_patient_note_info(sample_patient_id, note_type="discharge")
# Radiology Notes
print_patient_note_info(sample_patient_id, note_type="radiology")
# Labs
print_patient_lab_info(sample_patient_id)

In [ ]:
clear_cache_directory(CACHE_DIR)